Fake_News_Detection_BERT.ipynb



In [ ]:
import pandas as pd

df = pd.read_csv('/content/FA-KES-Dataset.csv', encoding='latin1')

print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(df.columns)

In [ ]:
df['labels'].value_counts()

In [ ]:
df = df[['article_content','labels']]
df.head()

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(example):
    return tokenizer(example["article_content"], padding="max_length", truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_steps=10
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)

print(preds[:10])

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_df["labels"], preds))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(test_df["labels"], preds)

sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
text = "Breaking news: scientists discovered life on Mars"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)

prediction = outputs.logits.argmax().item()

if prediction == 1:
    print("Fake News")
else:
    print("Real News")

In [ ]:
model.save_pretrained("fake_news_model")
tokenizer.save_pretrained("fake_news_model")

In [ ]:
!ls

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
model.save_pretrained("fake_news_model")
tokenizer.save_pretrained("fake_news_model")

In [ ]:
!ls

In [ ]:
!zip -r fake_news_model.zip fake_news_model

In [ ]:
from google.colab import files
files.download("fake_news_model.zip")

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)
preds = predictions.predictions.argmax(-1)

print(classification_report(val_labels, preds))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(val_labels, preds)

sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
for i in range(20):
    if preds[i] != val_labels.iloc[i]:
        print("Text:", val_texts.iloc[i])
        print("Actual:", val_labels.iloc[i])
        print("Predicted:", preds[i])
        print()

Error Analysis

Example 1:
Text: Government announces new economic policy
Actual Label: Real
Predicted Label: Fake
Reason: The model confused because the headline contains words often used in fake news.

Example 2:
Text: Celebrity found alive after death rumor
Actual Label: Fake
Predicted Label: Real
Reason: The model misclassified because the headline structure looks like real news.

Example 3:
Text: New study shows health benefits of coffee
Actual Label: Real
Predicted Label: Fake
Reason: The model predicted fake because of sensational wording.

Example 4:
Text: Viral post claims moon will disappear tonight
Actual Label: Fake
Predicted Label: Real
Reason: The model failed to detect exaggeration.

Example 5:
Text: Government passes new law for education reform
Actual Label: Real
Predicted Label: Fake
Reason: The model misclassified due to similarity with political fake news headlines.